# Findings Summary: Notebooks 007-013 — Window Correction, Sliding-Window Discovery, and Multi-Rat Validation

**Scope of this notebook:** the advisor feedback response and everything that followed from it, a
sampling-rate bug fix, the discovery that the original fixed window was substantially suboptimal, and
the multi-round validation process that followed. All plots in this notebook use the exact numeric
results already produced and reviewed in notebooks 007-012, this notebook doesn't recompute anything,
it renders the same numbers into a single, presentable document.

**One-line summary of what changed across this arc:** we found and fixed a real timing bug, discovered
the window we'd been using was substantially suboptimal, validated a better one across all 5 rats with
increasing rigor (single-run -> repeated cross-validation -> extended per-rat safe span), and caught a
data-structure issue (inconsistent channel counts across rats) before it could cause a silent bug in
future pooled modeling.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 1. The Bug: Non-Uniform Sampling Rate (Notebook 007)

While responding to advisor feedback about window definition clarity, comparing a trial's real timestamp
against its sample index revealed the session's sampling rate is NOT constant, instantaneous rate ran
close to 1000 Hz early in the Mitt session, while the session-wide average (used everywhere before this
point to size windows) was 714.05 Hz. Windows sized from that single average did not reliably span the
intended 500ms of real time throughout the recording.

**Fix:** window edges located from each trial's own real timestamp (`TimeBin`) directly, via
`np.searchsorted`, rather than an assumed constant rate.

**Result of the fix, tested on Mitt:** InSeq/OutSeq balanced accuracy 55.9% (old, average-rate windowing)
vs. 55.5% (corrected, timestamp-precise windowing), a negligible difference for this specific session.
**Conclusion:** the bug was real and worth fixing for correctness and defensibility, but it was not the
main source of the accuracy gap to the ~85% target, that turned out to be something else entirely (see
Section 2).


## 2. The Real Discovery: The Fixed 500ms Window Was Suboptimal (Notebook 008)

Rather than assuming a 500ms window starting exactly at Poke-In was correct, we tested it directly: a
250ms window slid from 500ms before Poke-In to 1500ms after, in 25ms steps, on Mitt. This is real data
from that scan, not a schematic.


In [ ]:
# Real data from notebook 008's full sliding-window scan on Mitt
offsets_ms_nb008 = np.array([-500,-475,-450,-425,-400,-375,-350,-325,-300,-275,-250,-225,-200,-175,-150,
    -125,-100,-75,-50,-25,0,25,50,75,100,125,150,175,200,225,250,275,300,325,350,375,400,425,450,475,
    500,525,550,575,600,625,650,675,700,725,750,775,800,825,850,875,900,925,950,975,1000,1025,1050,
    1075,1100,1125,1150,1175,1200,1225,1250,1275,1300,1325,1350,1375,1400,1425,1450,1475,1500])

inseq_curve_nb008 = np.array([0.5328,0.4542,0.5008,0.5104,0.5161,0.4784,0.4956,0.4805,0.4932,0.4921,
    0.4414,0.4206,0.4209,0.4789,0.4654,0.479,0.5366,0.5122,0.5231,0.4935,0.605,0.5063,0.4841,0.4988,
    0.4931,0.4933,0.4826,0.5585,0.4669,0.4709,0.5324,0.5403,0.4896,0.5922,0.5233,0.568,0.4936,0.4884,
    0.5038,0.5147,0.5683,0.6019,0.5705,0.5889,0.6109,0.555,0.6015,0.5681,0.587,0.5646,0.5608,0.5113,
    0.5814,0.5832,0.6074,0.63,0.6205,0.6373,0.5852,0.6171,0.5738,0.5595,0.665,0.6448,0.5942,0.6505,
    0.6281,0.5872,0.5967,0.6133,0.6671,0.6035,0.6819,0.6818,0.639,0.6481,0.6224,0.5775,0.6134,0.6952,
    0.6576])

odor_curve_nb008 = np.array([0.2591,0.2893,0.2733,0.2969,0.3159,0.2827,0.2976,0.2837,0.2747,0.2946,
    0.2472,0.2234,0.2716,0.2942,0.2906,0.3184,0.3243,0.2921,0.3334,0.2966,0.286,0.2992,0.303,0.2816,
    0.2376,0.2311,0.2585,0.2765,0.2867,0.3151,0.2661,0.317,0.2858,0.3418,0.3327,0.34,0.3852,0.3622,
    0.3407,0.3645,0.3179,0.3513,0.3603,0.3416,0.3054,0.3301,0.3607,0.3201,0.2918,0.3329,0.3571,0.3709,
    0.349,0.3601,0.3262,0.3358,0.3516,0.3713,0.3592,0.3479,0.3475,0.4279,0.3496,0.3143,0.3339,0.2948,
    0.2851,0.301,0.2986,0.3334,0.3414,0.3317,0.3388,0.2976,0.2855,0.2622,0.3055,0.2739,0.2761,0.2827,
    0.2742])

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(offsets_ms_nb008, inseq_curve_nb008, label='InSeq/OutSeq', color='#4C72B0', linewidth=2)
ax.plot(offsets_ms_nb008, odor_curve_nb008, label='Odor Identity', color='#DD8452', linewidth=2)
ax.axhline(0.5, color='#4C72B0', linestyle='--', alpha=0.4, label='InSeq/OutSeq chance (0.5)')
ax.axhline(0.2, color='#DD8452', linestyle='--', alpha=0.4, label='Odor chance (0.2)')
ax.axvline(0, color='black', linestyle=':', alpha=0.6, label='Poke-In (t=0)')
ax.axvline(500, color='green', linestyle='-', alpha=0.4, label='old fixed window end (500ms)')
ax.set_xlabel('Window start time relative to Poke-In (ms)')
ax.set_ylabel('Balanced accuracy (5-fold CV)')
ax.set_title('Sliding-window decoding curve, Mitt (notebook 008, first discovery)')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

print(f"Fixed 500ms window result: 55.5%")
print(f"Best single-run result found by sliding: {inseq_curve_nb008.max()*100:.1f}% at offset {offsets_ms_nb008[inseq_curve_nb008.argmax()]}ms")


**This single plot is the project's central finding.** Everything from this point forward is about
validating whether this pattern (later windows are substantially better) is real and generalizes, not
just a lucky result on one rat.


## 3. First Multi-Rat Check (Notebook 009): Promising, But Untrustworthy As-Is

The same scan repeated independently on all 5 rats found 3 of 5 rats near or above the ~85% target. But
a red flag appeared: 3 of 5 rats picked their single "best" Odor Identity window from BEFORE Poke-In,
when the odor hadn't been presented yet, a sign that picking one maximum out of many tested positions
was partly selecting noise, not just signal.


In [ ]:
rats = ['Mitt', 'Barat', 'Stella', 'Superchris', 'Buchanan']
nb009_inseq = [0.6952, 0.8098, 0.8065, 0.8976, 0.7126]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(rats, nb009_inseq, color='#C44E52')
ax.axhline(0.85, color='green', linestyle='--', label='target (~0.85)')
ax.set_ylabel('Single-run best balanced accuracy')
ax.set_title('Notebook 009: single-run peak, InSeq/OutSeq (before validation)')
ax.legend()
plt.tight_layout()
plt.show()


## 4. Repeated Cross-Validation Correction (Notebook 010)

Re-evaluating with `RepeatedStratifiedKFold` (25 scores per window position instead of 5) to separate
real signal from search noise. The comparison below shows exactly how much the single-run numbers moved
once evaluated more rigorously.


In [ ]:
nb010_inseq = [0.7005, 0.7307, 0.7984, 0.8867, 0.7071]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(rats))
width = 0.35
ax.bar(x - width/2, nb009_inseq, width, label='Single-run (notebook 009)', color='#C44E52', alpha=0.7)
ax.bar(x + width/2, nb010_inseq, width, label='Repeated CV, 25 folds (notebook 010)', color='#4C72B0')
ax.axhline(0.85, color='green', linestyle='--', label='target (~0.85)')
ax.set_xticks(x)
ax.set_xticklabels(rats)
ax.set_ylabel('Balanced accuracy')
ax.set_title('Effect of repeated cross-validation on reported InSeq/OutSeq accuracy')
ax.legend()
plt.tight_layout()
plt.show()

print("Barat dropped the most (81.0% -> 73.1%), confirming its single-run number was substantially")
print("inflated by search noise. Superchris and Buchanan barely moved, their strong/weak results held up.")


**Signal-quality side-finding from notebook 010:** Superchris's raw signal RMS amplitude (0.269) was
nearly double every other rat's (0.129-0.161), a plausible partial explanation for its consistently
strong decoding performance, investigated further in Section 6.


## 5. Extended Per-Rat Safe-Span Validation (Notebook 011)

Two remaining questions: were Barat and Stella's peaks (both landing at the very edge of the tested
range) true peaks, or just where the search stopped? And was Superchris's amplitude difference driven by
a single outlier channel? Each rat's search span was extended to THAT RAT'S OWN safe limit, computed
from its real minimum trial-to-trial gap (measured directly from timestamps) minus the window length
minus a 100ms safety buffer, so no window could ever reach into the next trial.


In [ ]:
min_gaps_ms = {'Mitt': 2768, 'Barat': 2311, 'Stella': 3167, 'Superchris': 2147, 'Buchanan': 2953}
spans_tested = {'Mitt': (-500, 2400), 'Barat': (-500, 1950), 'Stella': (-500, 2800),
                'Superchris': (-500, 1750), 'Buchanan': (-500, 2600)}
final_peaks = {'Mitt': (0.7592, 2350), 'Barat': (0.7982, 1700), 'Stella': (0.7984, 1500),
               'Superchris': (0.8867, 1000), 'Buchanan': (0.7119, 1700)}

fig, ax = plt.subplots(figsize=(11, 5))
for i, rat in enumerate(rats):
    span = spans_tested[rat]
    peak_offset = final_peaks[rat][1]
    ax.plot(span, [i, i], color='#4C72B0', linewidth=6, alpha=0.4, solid_capstyle='butt')
    ax.plot(peak_offset, i, 'o', color='#C44E52', markersize=10, zorder=5)
    ax.text(span[1] + 60, i, f"{final_peaks[rat][0]*100:.1f}%", va='center', fontsize=9)

ax.axvline(0, color='black', linestyle=':', alpha=0.6, label='Poke-In (t=0)')
ax.set_yticks(range(len(rats)))
ax.set_yticklabels(rats)
ax.set_xlabel('Offset from Poke-In (ms)')
ax.set_title('Each rat\'s tested span (blue bar) and confirmed interior peak (red dot)')
ax.legend()
plt.tight_layout()
plt.show()

print("All 5 peaks confirmed INTERIOR to their tested span, not sitting at the boundary,")
print("meaning these are real, found peaks, not artifacts of stopping the search too early.")


## 6. Superchris's Amplitude: Not a Single Bad Channel

Per-channel RMS breakdown found 0 of 21 channels exceeded 2 standard deviations above the mean (max-to-
min ratio 2.65x, but spread continuously, not concentrated in one outlier). A visual raw-trace comparison
against Mitt (same channel, same voltage scale) supported this being a genuinely different recording
rather than one faulty electrode. Note: Superchris has 21 channels, not 22, addressed in Section 7.


## 7. Cross-Rat Channel Structure Audit (Notebook 012)

Closing a gap left by the earlier structural audit (notebook 02), which only checked behavioral channel
names, never LFP channel identity or count. Result: channel gaps are real, but rat-specific, not a
processing error.

| Rat | Missing LFP Channel(s) |
|---|---|
| Mitt | none |
| Barat | none |
| Stella | T1 |
| Superchris | T17 |
| Buchanan | T3, T14 |

**Interpretation:** each animal is missing a DIFFERENT channel (not a consistent pattern like "always the
last channel"), which is exactly what you'd expect from real per-animal implant variation (a wire that
didn't yield usable signal for that specific surgery), not a data processing bug. **Practical
consequence:** any code that assumes a fixed channel count, or that channel position (not name) is
consistent across rats, needs to be corrected before pooling data across rats.


## 8. Final Validated Results, All 5 Rats


In [ ]:
inseq_final = [0.759, 0.798, 0.798, 0.887, 0.712]
odor_final = [0.381, 0.433, 0.474, 0.431, 0.464]

fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(rats))
width = 0.35
bars1 = ax.bar(x - width/2, inseq_final, width, label='InSeq/OutSeq (chance=0.5)', color='#4C72B0')
bars2 = ax.bar(x + width/2, odor_final, width, label='Odor Identity (chance=0.2)', color='#DD8452')
ax.axhline(0.85, color='green', linestyle='--', alpha=0.6, label='target (~0.85)')
ax.set_xticks(x)
ax.set_xticklabels(rats)
ax.set_ylabel('Validated peak balanced accuracy')
ax.set_title('Final validated results, all 5 rats (extended-span, repeated cross-validation)')
ax.legend()
for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.annotate(f'{h:.2f}', xy=(bar.get_x() + bar.get_width()/2, h), xytext=(0, 3),
                textcoords='offset points', ha='center', fontsize=9)
plt.tight_layout()
plt.show()


## 9. Open Items (Not Yet Done)

- Multi-rat pooled training (session-based split), not yet built.
- Permutation test to formally confirm the strongest result (Superchris, 88.7%) is statistically
  distinguishable from chance.
- RNN retraining with the validated window (only the GLM has been retrained at the new window so far).
- Pipeline update to read each rat's actual channel list rather than assuming a fixed count, given
  Section 7's findings.
- Confirmation with the lab on whether the per-rat channel gaps reflect known hardware issues.

## 10. Recommended Presentation Narrative

1. Project goal and MaGNet benchmark.
2. The two bugs found and fixed (RNN class imbalance collapse, sampling-rate windowing error), evidence
   of a careful, validated process.
3. The central discovery (Section 2's plot): window choice mattered far more than assumed.
4. The validation arc (Sections 3-5): single-run -> repeated CV -> extended per-rat span, each step
   catching and correcting a real issue.
5. Final validated results table (Section 8) as the headline, with honest caveats: Mitt weakest, odor
   has no sharp temporal peak, Superchris's channel count difference.
6. Open items as the explicit near-term plan.
